# Alignment of Macroeconomic and Geopolitical Datasets

<p><i>Author:</i> Angelica Vanti</p>

<p><i>Project:</i> Predicting EUR/USD Movements Using Geopolitical and Macroeconomic Variables</p>

This notebook aligns the stationary macroeconomic and financial variables with the five alternative daily geopolitical-risk specifications constructed from LLM-scored tweets.

The objective is to create a common modelling sample in which the baseline macroeconomic specification and all geopolitical extensions contain identical macroeconomic observations and dates. This ensures that differences in subsequent forecasting performance are attributable to the inclusion and representation of geopolitical information rather than differences in sample composition.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

# Inputs
MACRO_PATH = PROCESSED_DIR / "macro_processed.csv"

GEO_MAX_PATH = PROCESSED_DIR / "geopolitical_indices_daily_max.csv"
GEO_SUM_PATH = PROCESSED_DIR / "geopolitical_indices_daily_sum.csv"
GEO_MA3_PATH = PROCESSED_DIR / "geopolitical_indices_daily_sum_ma3.csv"
GEO_MA5_PATH = PROCESSED_DIR / "geopolitical_indices_daily_sum_ma5.csv"
GEO_MA10_PATH = PROCESSED_DIR / "geopolitical_indices_daily_sum_ma10.csv"

# Outputs
BASELINE_COMMON_PATH = PROCESSED_DIR / "macro_baseline_common_sample.csv"

OUT_MAX_PATH = PROCESSED_DIR / "macro_geopolitical_max.csv"
OUT_SUM_PATH = PROCESSED_DIR / "macro_geopolitical_sum.csv"
OUT_MA3_PATH = PROCESSED_DIR / "macro_geopolitical_ma3.csv"
OUT_MA5_PATH = PROCESSED_DIR / "macro_geopolitical_ma5.csv"
OUT_MA10_PATH = PROCESSED_DIR / "macro_geopolitical_ma10.csv"

In [3]:
macro = pd.read_csv(MACRO_PATH)

geo_max = pd.read_csv(GEO_MAX_PATH)
geo_sum = pd.read_csv(GEO_SUM_PATH)
geo_ma3 = pd.read_csv(GEO_MA3_PATH)
geo_ma5 = pd.read_csv(GEO_MA5_PATH)
geo_ma10 = pd.read_csv(GEO_MA10_PATH)

print("Macro:", macro.shape)
print("MAX:", geo_max.shape)
print("SUM:", geo_sum.shape)
print("MA3:", geo_ma3.shape)
print("MA5:", geo_ma5.shape)
print("MA10:", geo_ma10.shape)

Macro: (2376, 5)
MAX: (1469, 4)
SUM: (1469, 4)
MA3: (1469, 4)
MA5: (1469, 4)
MA10: (1469, 4)


## 1. Data Loading and Sample Overlap

The processed macroeconomic dataset and the five geopolitical index specifications are loaded and their date variables converted to a common datetime format.

The geopolitical sample ends on 8 January 2021. The macroeconomic dataset extends beyond this period, so only macro observations up to the end of the geopolitical sample are retained for the alignment procedure.

In [4]:
# Convert all date columns to datetime
macro["observation_date"] = pd.to_datetime(
    macro["observation_date"],
    errors="coerce"
)

for df in [geo_max, geo_sum, geo_ma3, geo_ma5, geo_ma10]:
    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

# Validate dates
assert macro["observation_date"].notna().all()

for name, df in {
    "MAX": geo_max,
    "SUM": geo_sum,
    "MA3": geo_ma3,
    "MA5": geo_ma5,
    "MA10": geo_ma10,
}.items():
    assert df["date"].notna().all(), f"{name} contains invalid dates."

# End of geopolitical sample
overlap_end = min(
    macro["observation_date"].max(),
    geo_sum["date"].max()
)

# Macro observations available during geopolitical sample
macro_overlap = macro[
    macro["observation_date"] <= overlap_end
].copy()

print("Macro date range:")
print(
    macro["observation_date"].min(),
    "to",
    macro["observation_date"].max()
)

print("\nGeopolitical date range:")
print(
    geo_sum["date"].min(),
    "to",
    geo_sum["date"].max()
)

print("\nMacro observations in geopolitical overlap:")
print(len(macro_overlap))

Macro date range:
2017-01-04 00:00:00 to 2026-07-24 00:00:00

Geopolitical date range:
2017-01-01 00:00:00 to 2021-01-08 00:00:00

Macro observations in geopolitical overlap:
997


In [5]:
# Available macro observation dates during the geopolitical sample
macro_dates = (
    macro_overlap[["observation_date"]]
    .sort_values("observation_date")
    .rename(columns={"observation_date": "macro_date"})
)


def map_to_next_macro_date(geo_df):
    """
    Assign each geopolitical calendar day to the next available
    macro observation date.

    Geopolitical information released during weekends, holidays,
    or other macro-data gaps is therefore incorporated into the
    next observed market date.
    """
    geo = geo_df.sort_values("date").copy()

    mapped = pd.merge_asof(
        geo,
        macro_dates,
        left_on="date",
        right_on="macro_date",
        direction="forward"
    )

    return mapped

## 2. Alignment of Daily MAX and SUM Indices

The tweet-based geopolitical indices are defined on a complete calendar-day basis, whereas the macroeconomic variables are observed only on available market/data dates.

For the daily MAX and SUM specifications, each geopolitical calendar day is therefore assigned to the next available macro observation date. Geopolitical information released during weekends, holidays or other gaps is consequently incorporated into the next observed market date.

Where multiple geopolitical calendar days map to the same macro date, SUM scores are added and MAX scores retain the largest assigned value.

In [6]:
# Test the alignment using the daily SUM series
sum_mapped = map_to_next_macro_date(geo_sum)

assert len(sum_mapped) == len(geo_sum)
assert sum_mapped["macro_date"].notna().all()

print("Original geopolitical rows:", len(geo_sum))
print("Mapped rows:", len(sum_mapped))
print(
    "Rows with no future macro date:",
    sum_mapped["macro_date"].isna().sum()
)

print("\nExample around a weekend:")
display(
    sum_mapped[
        (sum_mapped["date"] >= "2017-01-06") &
        (sum_mapped["date"] <= "2017-01-10")
    ]
)

Original geopolitical rows: 1469
Mapped rows: 1469
Rows with no future macro date: 0

Example around a weekend:


,date,trade_sum,sanctions_sum,fed_pressure_sum,macro_date
5,2017-01-06,40.0,0.0,0.0,2017-01-06
6,2017-01-07,0.0,0.0,0.0,2017-01-09
7,2017-01-08,0.0,0.0,0.0,2017-01-09
8,2017-01-09,105.0,25.0,0.0,2017-01-09
9,2017-01-10,0.0,0.0,0.0,2017-01-10


In [7]:
# Aggregate SUM and MAX onto macro dates

# Map both geopolitical specifications
sum_mapped = map_to_next_macro_date(geo_sum)
max_mapped = map_to_next_macro_date(geo_max)

# SUM:
# add all geopolitical activity assigned to each available macro date
sum_aligned = (
    sum_mapped
    .groupby("macro_date")[
        ["trade_sum", "sanctions_sum", "fed_pressure_sum"]
    ]
    .sum()
    .reset_index()
)

# MAX:
# keep the largest geopolitical score assigned to each available macro date
max_aligned = (
    max_mapped
    .groupby("macro_date")[
        ["trade_max", "sanctions_max", "fed_pressure_max"]
    ]
    .max()
    .reset_index()
)

print("Aligned SUM rows:", len(sum_aligned))
print("Aligned MAX rows:", len(max_aligned))

print("\nSUM example:")
display(
    sum_aligned[
        (sum_aligned["macro_date"] >= "2017-01-06") &
        (sum_aligned["macro_date"] <= "2017-01-10")
    ]
)

print("\nMAX example:")
display(
    max_aligned[
        (max_aligned["macro_date"] >= "2017-01-06") &
        (max_aligned["macro_date"] <= "2017-01-10")
    ]
)

Aligned SUM rows: 997
Aligned MAX rows: 997

SUM example:


,macro_date,trade_sum,sanctions_sum,fed_pressure_sum
2,2017-01-06,40.0,0.0,0.0
3,2017-01-09,105.0,25.0,0.0
4,2017-01-10,0.0,0.0,0.0



MAX example:


,macro_date,trade_max,sanctions_max,fed_pressure_max
2,2017-01-06,40.0,0.0,0.0
3,2017-01-09,65.0,25.0,0.0
4,2017-01-10,0.0,0.0,0.0


## 3. Alignment of Moving-Average Indices

The 3-, 5-, and 10-day moving-average geopolitical indices already incorporate information from preceding calendar days. These series are therefore aligned directly to matching macroeconomic observation dates rather than being carried forward again.

This avoids applying an additional persistence mechanism to variables that are already temporally smoothed.

In [8]:
# Align MA3, MA5, and MA10 series to macro dates

def align_moving_average_to_macro(geo_ma):
    """
    Align an already constructed calendar-day moving-average
    geopolitical index to available macro observation dates.

    No extra carry-forward is applied because the moving average
    already contains information from preceding calendar days.
    """
    aligned = macro_overlap[["observation_date"]].merge(
        geo_ma,
        left_on="observation_date",
        right_on="date",
        how="left"
    )

    return aligned.drop(columns="date")


ma3_aligned = align_moving_average_to_macro(geo_ma3)
ma5_aligned = align_moving_average_to_macro(geo_ma5)
ma10_aligned = align_moving_average_to_macro(geo_ma10)

print("MA3 aligned:", ma3_aligned.shape)
print("MA5 aligned:", ma5_aligned.shape)
print("MA10 aligned:", ma10_aligned.shape)

print("\nMissing values:")
print("\nMA3:")
print(ma3_aligned.isna().sum())

print("\nMA5:")
print(ma5_aligned.isna().sum())

print("\nMA10:")
print(ma10_aligned.isna().sum())



MA3 aligned: (997, 4)
MA5 aligned: (997, 4)
MA10 aligned: (997, 4)

Missing values:

MA3:
observation_date        0
trade_sum_ma3           0
sanctions_sum_ma3       0
fed_pressure_sum_ma3    0
dtype: int64

MA5:
observation_date        0
trade_sum_ma5           1
sanctions_sum_ma5       1
fed_pressure_sum_ma5    1
dtype: int64

MA10:
observation_date         0
trade_sum_ma10           4
sanctions_sum_ma10       4
fed_pressure_sum_ma10    4
dtype: int64


## 4. Construction of a Common Modelling Sample

To ensure that all subsequent model comparisons use exactly the same observations, the final sample is restricted to dates for which the longest geopolitical moving-average specification, MA(10), is fully available.

This produces a common sample from **10 January 2017 to 8 January 2021**, containing **993 macroeconomic observations**.

The macro-only baseline and all five geopolitical specifications are subsequently constructed on these identical dates.

In [9]:
# Use dates for which the longest moving average is fully available
common_dates = ma10_aligned.loc[
    ma10_aligned[
        [
            "trade_sum_ma10",
            "sanctions_sum_ma10",
            "fed_pressure_sum_ma10",
        ]
    ].notna().all(axis=1),
    "observation_date"
]

COMMON_START = common_dates.min()
COMMON_END = common_dates.max()

print("Common sample start:", COMMON_START)
print("Common sample end:", COMMON_END)
print("Common macro observations:", len(common_dates))

Common sample start: 2017-01-10 00:00:00
Common sample end: 2021-01-08 00:00:00
Common macro observations: 993


In [10]:
macro_common = (
    macro_overlap[
        macro_overlap["observation_date"].isin(common_dates)
    ]
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print("Macro baseline:", macro_common.shape)
print(
    "Date range:",
    macro_common["observation_date"].min(),
    "to",
    macro_common["observation_date"].max()
)

Macro baseline: (993, 5)
Date range: 2017-01-10 00:00:00 to 2021-01-08 00:00:00


In [11]:
# MAX
macro_geo_max = (
    macro_common
    .merge(
        max_aligned,
        left_on="observation_date",
        right_on="macro_date",
        how="left"
    )
    .drop(columns="macro_date")
)

# SUM
macro_geo_sum = (
    macro_common
    .merge(
        sum_aligned,
        left_on="observation_date",
        right_on="macro_date",
        how="left"
    )
    .drop(columns="macro_date")
)

# MA(3)
macro_geo_ma3 = macro_common.merge(
    ma3_aligned,
    on="observation_date",
    how="left"
)

# MA(5)
macro_geo_ma5 = macro_common.merge(
    ma5_aligned,
    on="observation_date",
    how="left"
)

# MA(10)
macro_geo_ma10 = macro_common.merge(
    ma10_aligned,
    on="observation_date",
    how="left"
)

print("Baseline:", macro_common.shape)
print("MAX:     ", macro_geo_max.shape)
print("SUM:     ", macro_geo_sum.shape)
print("MA(3):   ", macro_geo_ma3.shape)
print("MA(5):   ", macro_geo_ma5.shape)
print("MA(10):  ", macro_geo_ma10.shape)

Baseline: (993, 5)
MAX:      (993, 8)
SUM:      (993, 8)
MA(3):    (993, 8)
MA(5):    (993, 8)
MA(10):   (993, 8)


## 5. Validation and Export

The six resulting modelling datasets are validated before export.

The checks confirm that:

- all specifications contain exactly 993 observations;
- no duplicate dates are present;
- no missing values remain;
- all six datasets use identical observation dates; and
- the four macroeconomic variables are numerically identical across every specification.

These checks ensure that subsequent differences in model performance cannot be attributed to differences in the underlying macroeconomic sample.

The validated datasets are then exported and reloaded to confirm that the saved files retain the expected dimensions and contain no missing or duplicate observations.

In [12]:
datasets = {
    "baseline": macro_common,
    "max": macro_geo_max,
    "sum": macro_geo_sum,
    "ma3": macro_geo_ma3,
    "ma5": macro_geo_ma5,
    "ma10": macro_geo_ma10,
}

MACRO_COLS = [
    "DEXUSEU_logreturn",
    "DGS2_diff",
    "USEPUINDXD_diff",
    "VIXCLS",
]

baseline_dates = macro_common["observation_date"].reset_index(drop=True)

for name, df in datasets.items():

    # Same sample size
    assert len(df) == 993, f"{name}: unexpected row count"

    # No duplicate dates
    assert not df["observation_date"].duplicated().any(), \
        f"{name}: duplicate dates found"

    # No missing values
    assert not df.isna().any().any(), \
        f"{name}: missing values found"

    # Identical dates
    assert df["observation_date"].reset_index(drop=True).equals(
        baseline_dates
    ), f"{name}: dates differ from baseline"

    # Identical macroeconomic data
    for col in MACRO_COLS:
        assert np.allclose(
            df[col].to_numpy(),
            macro_common[col].to_numpy()
        ), f"{name}: {col} differs from baseline"

print("All six modelling specifications passed validation.")

All six modelling specifications passed validation.


In [13]:
macro_common.to_csv(
    BASELINE_COMMON_PATH,
    index=False
)

macro_geo_max.to_csv(
    OUT_MAX_PATH,
    index=False
)

macro_geo_sum.to_csv(
    OUT_SUM_PATH,
    index=False
)

macro_geo_ma3.to_csv(
    OUT_MA3_PATH,
    index=False
)

macro_geo_ma5.to_csv(
    OUT_MA5_PATH,
    index=False
)

macro_geo_ma10.to_csv(
    OUT_MA10_PATH,
    index=False
)

print("Saved:")
print(BASELINE_COMMON_PATH)
print(OUT_MAX_PATH)
print(OUT_SUM_PATH)
print(OUT_MA3_PATH)
print(OUT_MA5_PATH)
print(OUT_MA10_PATH)

Saved:
..\data\processed\macro_baseline_common_sample.csv
..\data\processed\macro_geopolitical_max.csv
..\data\processed\macro_geopolitical_sum.csv
..\data\processed\macro_geopolitical_ma3.csv
..\data\processed\macro_geopolitical_ma5.csv
..\data\processed\macro_geopolitical_ma10.csv


In [14]:
saved_outputs = {
    "baseline": BASELINE_COMMON_PATH,
    "max": OUT_MAX_PATH,
    "sum": OUT_SUM_PATH,
    "ma3": OUT_MA3_PATH,
    "ma5": OUT_MA5_PATH,
    "ma10": OUT_MA10_PATH,
}

for name, path in saved_outputs.items():

    df = pd.read_csv(path)

    assert len(df) == 993
    assert not df.isna().any().any()
    assert not df["observation_date"].duplicated().any()

    print(
        f"{name:8s}: "
        f"{df.shape[0]} rows × {df.shape[1]} columns"
    )

print("\nAll saved files successfully reloaded and validated.")

baseline: 993 rows × 5 columns
max     : 993 rows × 8 columns
sum     : 993 rows × 8 columns
ma3     : 993 rows × 8 columns
ma5     : 993 rows × 8 columns
ma10    : 993 rows × 8 columns

All saved files successfully reloaded and validated.


## 6. Summary

This notebook constructed a common modelling sample combining the stationary macroeconomic variables with five alternative representations of the LLM-derived geopolitical-risk indices.

Daily MAX and SUM scores were mapped to the next available macroeconomic observation date, while the already-smoothed MA(3), MA(5), and MA(10) indices were aligned directly to observed macro dates.

The final sample was restricted to the period over which MA(10) was fully available, yielding 993 common observations between 10 January 2017 and 8 January 2021. A macro-only baseline and five geopolitical model datasets were then created on exactly the same dates and validated to contain identical macroeconomic observations.

These datasets form the direct inputs to the subsequent VAR-X model comparison.